# ProteinNet Dataset Onboarding (CASP12)

This notebook does **one job**: get ProteinNet onto your machine/cluster and convert it into a format that trains fast.

**What you’ll end up with:**
- `data/raw/casp12/...` (original ProteinNet split files)
- `data/processed/casp12/...` (memory-mapped `.npy` arrays used by `train.py`)

> If you're running on SOL and don't want to use Jupyter, you can copy/paste the same commands into the terminal.

---

## 0) Pick directories

On an HPC cluster you typically want:
- **raw** download/extract in a scratch area
- **processed** arrays also in scratch (training reads them constantly)

Set paths below to match your environment.


In [11]:
from pathlib import Path

# Edit these if needed
RAW_ROOT = Path("data/raw")
PROC_ROOT = Path("data/processed")

RAW_ROOT.mkdir(parents=True, exist_ok=True)
PROC_ROOT.mkdir(parents=True, exist_ok=True)

RAW_TAR = RAW_ROOT / "casp12.tar.gz"
RAW_EXTRACT_DIR = RAW_ROOT  # tar contains a casp12/ folder inside

print("RAW_ROOT:", RAW_ROOT.resolve())
print("PROC_ROOT:", PROC_ROOT.resolve())


RAW_ROOT: /scratch/mtiwar26/notebooks/data/raw
PROC_ROOT: /scratch/mtiwar26/notebooks/data/processed


## 1) Download ProteinNet CASP12 (human readable)

Official link from ProteinNet:
- https://sharehost.hms.harvard.edu/sysbio/alquraishi/proteinnet/human_readable/casp12.tar.gz

This file is large, so we use `wget -c` (continue mode).


In [ ]:
import subprocess, shlex

url = "https://sharehost.hms.harvard.edu/sysbio/alquraishi/proteinnet/human_readable/casp12.tar.gz"
cmd = f"wget -c {shlex.quote(url)} -O {shlex.quote(str(RAW_TAR))}"
print(cmd)

# Uncomment to run:
subprocess.run(cmd, shell=True, check=True)


wget -c https://sharehost.hms.harvard.edu/sysbio/alquraishi/proteinnet/human_readable/casp12.tar.gz -O data/raw/casp12.tar.gz


--2026-02-23 23:51:29--  https://sharehost.hms.harvard.edu/sysbio/alquraishi/proteinnet/human_readable/casp12.tar.gz
Resolving sharehost.hms.harvard.edu (sharehost.hms.harvard.edu)... 134.174.159.103
Connecting to sharehost.hms.harvard.edu (sharehost.hms.harvard.edu)|134.174.159.103|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14148108241 (13G) [application/x-gzip]
Saving to: ‘data/raw/casp12.tar.gz’

     0K .......... .......... .......... .......... ..........  0%  404K 9h29m
    50K .......... .......... .......... .......... ..........  0%  819K 7h5m
   100K .......... .......... .......... .......... ..........  0% 32.1M 4h46m
   150K .......... .......... .......... .......... ..........  0% 40.2M 3h35m
   200K .......... .......... .......... .......... ..........  0%  839K 3h47m
   250K .......... .......... .......... .......... ..........  0% 27.2M 3h11m
   300K .......... .......... .......... .......... ..........  0% 35.1M 2h44m
   350K ......

## 2) Extract

After extraction, you should have:
```
data/raw/casp12/
  training_90
  validation
  testing
  ...
```


In [3]:
cmd = f"tar -xzf {shlex.quote(str(RAW_TAR))} -C {shlex.quote(str(RAW_EXTRACT_DIR))}"
print(cmd)

# Uncomment to run:
subprocess.run(cmd, shell=True, check=True)

# Quick check (after extraction)
casp_dir = RAW_ROOT / "casp12"
print("Exists?", casp_dir.exists())
if casp_dir.exists():
    print("Files:", [p.name for p in casp_dir.iterdir()][:10], "...")


tar -xzf data/raw/casp12.tar.gz -C data/raw
Exists? False


## 3) Preprocess into fast memory-mapped arrays

This step reads the raw text records once and writes:
- `tokens.npy` (int16)
- `evo.npy` (float16)
- `ca_coords.npy` (float32, Å)
- `mask.npy` (uint8)
- `lengths.npy`, `offsets.npy`, `ids.txt`

You can tune `--max_length` to cap long proteins (speed / memory).


In [4]:
# From repo root, run:
# python scripts/preprocess_proteinnet.py --casp casp12 --raw_root data/raw --out_root data/processed --train_threshold 90 --max_length 1024

cmd = "python scripts/preprocess_proteinnet.py --casp casp12 --raw_root data/raw --out_root data/processed --train_threshold 90 --max_length 1024"
print(cmd)

# Uncomment to run:
subprocess.run(cmd, shell=True, check=True)


python scripts/preprocess_proteinnet.py --casp casp12 --raw_root data/raw --out_root data/processed --train_threshold 90 --max_length 1024


## 4) Sanity check processed data

This is optional, but it’s a nice “did the pipeline work?” check.


In [5]:
import numpy as np

train_dir = PROC_ROOT / "casp12" / "train_90"
print("Train dir exists?", train_dir.exists())

if train_dir.exists():
    tokens = np.load(train_dir / "tokens.npy", mmap_mode="r")
    evo = np.load(train_dir / "evo.npy", mmap_mode="r")
    ca = np.load(train_dir / "ca_coords.npy", mmap_mode="r")
    mask = np.load(train_dir / "mask.npy", mmap_mode="r")
    lengths = np.load(train_dir / "lengths.npy", mmap_mode="r")
    offsets = np.load(train_dir / "offsets.npy", mmap_mode="r")

    print("tokens:", tokens.shape, tokens.dtype)
    print("evo:", evo.shape, evo.dtype)
    print("ca:", ca.shape, ca.dtype)
    print("mask:", mask.shape, mask.dtype)
    print("records:", lengths.shape[0])
    print("max length:", int(lengths.max()))
    print("mean length:", float(lengths.mean()))

    # Grab one example
    i = 0
    s, e = int(offsets[i]), int(offsets[i+1])
    print("Example 0 length:", e - s)
    print("Example 0 tokens (first 20):", tokens[s:s+20].tolist())


Train dir exists? False
